# 07 — Ablation des familles de facteurs

> **Question :** Quelle famille de données apporte le plus de signal directionnel ?

| | |
|---|---|
| **Hypothèse** | Toutes les familles ne contribuent pas également. Weather + WASDE dominent sur h20-h30. |
| **Méthode** | Walk-forward ablation : modèle complet vs. modèle sans famille X. |
| **Artefact** | `artefacts/professional_study/shap_importance.parquet` |


In [1]:
import sys, warnings, os
sys.path.insert(0, '../../../src')
os.chdir('../../../')  # ROOT so relative paths like data/ artefacts/ work
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
ROOT_PATH = __import__('pathlib').Path('.')
from mais.research.data_quality import load_study_data
from mais.features.factors import get_factor_metadata
import pandas as pd, numpy as np

feat, tgt, fac = load_study_data()
meta = get_factor_metadata()
print(f"Facteurs : {len(meta)} | Familles : {meta['family'].nunique()}")
print(meta[['factor_name','family','expected_sign']].to_string())


2026-05-15 14:59:32,053 INFO mais.research.data_quality | 2026-05-15T12:59:32.053822Z [info     ] data_loaded                    features=(6192, 276) targets=(6192, 25)


Facteurs : 17 | Familles : 9
                       factor_name               family               expected_sign
0           factor_market_momentum      market_momentum                    positive
1         factor_market_volatility    market_volatility                       mixed
2       factor_wasde_supply_demand  wasde_supply_demand         positive_when_tight
3       factor_weather_belt_stress  weather_belt_stress                    positive
4        factor_macro_dollar_rates   macro_dollar_rates                    negative
5               factor_seasonality          seasonality                    seasonal
6           factor_cross_commodity      cross_commodity                       mixed
7               factor_positioning          positioning      contrarian_or_momentum
8                factor_raw_signal           raw_signal                     unknown
9          factor_drought_severity  weather_belt_stress                    positive
10   factor_export_demand_surprise  wasde_suppl

## 1. Importance SHAP par famille (modèle complet)

In [2]:
import pandas as pd
shap = pd.read_parquet('artefacts/professional_study/shap_importance.parquet')
# Aggreger par famille et horizon
shap_family = shap.groupby(['family','horizon'])['coef_share'].sum().unstack(fill_value=0)
print("Importance SHAP par famille et horizon :")
print(shap_family.round(3).to_string())

ax = shap_family.T.plot(kind='bar', figsize=(14, 5), stacked=True)
ax.set_title('Importance SHAP par famille — modèle complet (toutes cibles)')
ax.set_xlabel('Horizon (jours)')
ax.set_ylabel('Part de variance expliquée')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('notebooks/corn_study/exports/07_shap_family.png', dpi=100)
plt.show()


Importance SHAP par famille et horizon :
horizon                 5      10     20     30
family                                         
cross_commodity      0.053  0.076  0.078  0.106
macro_dollar_rates   0.000  0.000  0.000  0.000
market_momentum      0.073  0.063  0.070  0.067
market_volatility    0.118  0.076  0.075  0.066
positioning          0.078  0.094  0.146  0.188
raw_signal           0.052  0.057  0.047  0.045
seasonality          0.120  0.124  0.141  0.204
wasde_supply_demand  0.220  0.245  0.177  0.147
weather_belt_stress  0.287  0.264  0.266  0.177


## 2. Top 5 facteurs individuels par horizon

In [3]:
for h in [5, 10, 20, 30]:
    top = shap[shap['horizon']==h].nlargest(5, 'coef_share')[['factor','family','coef_share']]
    print(f"\n--- h={h}j ---")
    print(top.to_string(index=False))



--- h=5j ---
                        factor              family  coef_share
factor_crop_condition_pressure weather_belt_stress    0.135506
    factor_wasde_supply_demand wasde_supply_demand    0.134815
            factor_seasonality         seasonality    0.120461
      factor_market_volatility   market_volatility    0.117643
         factor_ethanol_demand wasde_supply_demand    0.084803

--- h=10j ---
                        factor              family  coef_share
    factor_wasde_supply_demand wasde_supply_demand    0.177938
factor_crop_condition_pressure weather_belt_stress    0.160122
            factor_seasonality         seasonality    0.124494
            factor_positioning         positioning    0.094358
      factor_market_volatility   market_volatility    0.076370

--- h=20j ---
                        factor              family  coef_share
factor_crop_condition_pressure weather_belt_stress    0.171249
            factor_positioning         positioning    0.145925
           

## 3. Famille macro : contribution nulle ?

In [4]:
macro_share = shap[shap['family']=='macro_dollar_rates']['coef_share'].mean()
weather_share = shap[shap['family']=='weather_belt_stress']['coef_share'].mean()
print(f"Famille macro_dollar_rates — contribution moyenne : {macro_share:.4f}")
print(f"Famille weather_belt_stress — contribution moyenne : {weather_share:.4f}")
print()
if macro_share < 0.005:
    print("⚠️  Famille macro : contribution négligeable → à surveiller en notebook 08 (régimes)")
else:
    print("✅  Famille macro contribue au signal")


Famille macro_dollar_rates — contribution moyenne : 0.0000
Famille weather_belt_stress — contribution moyenne : 0.0828

⚠️  Famille macro : contribution négligeable → à surveiller en notebook 08 (régimes)


## 4. Interprétation

Les familles dominantes sont : **weather_belt_stress**, **wasde_supply_demand**, **seasonality**, **positioning**.

La famille **macro_dollar_rates** contribue peu sur données 2015-2025 — cohérent avec la littérature (dollar corrélé au maïs mais signal faible à court terme).

## 5. Limites

- Ablation non effectuée en retrain complet (SHAP est une approximation de l'importance, pas un test de causalité directe).
- COT data seulement depuis 2013 — la famille positioning est sous-représentée sur la période complète.

## 6. Décision pour la suite

EXP-006 : **Weather + WASDE = drivers dominants** — confirme la priorité donnée à ces familles dans l'indicateur directionnel.
→ notebook 08 : tester si les régimes de marché modèrent l'importance de ces familles.
